In [1]:
import numpy as np
import jax.numpy as jnp
from splank import SoftPlankton

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Assembly creation

In [2]:
yaml_data = """
# Variable Prefixes (Optional)
dof_names:        # DOF prefixes/names (extends default "dof")
    - x           # Recognized as a DOF prefix/name (e.g., "x0" in expressions)
param_names:      # Parameter prefixes/names (extends default "param" and "k")
    - myradius    # Recognized as a parameter prefix/name (e.g., `myradius` in expressions)
    - distance    # Recognized as a parameter prefix/name (e.g., `distance` in expressions)

# Default Values (Optional)
defaults:
  x0: 0           # DOF: Matches listed "x" prefix
  myradius: 0.25  # Parameter: Listed in `param_names`
  distance: 0     # Parameter: Listed in `param_names`
  k: 1            # Parameter: Auto-detected (default prefix "k")

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: myradius                       # Uses parameter `myradius`
    mass: 1
    position: [0, 0, distance]             # Explicit coordinates (unquoted numbers)
    orientation: [x0, x1, 0]               # Uses DOF `x0` (default "dof" prefix)
    torque: [-k*x0, -k*x1, 0]              # Mix of parameter (`k`) and DOFs (`x0`, `x1`)

  - # Sphere 1 #################
    radius: "1 - myradius"                 # Expression (quotes optional)
    mass: -1
    position: [0, 0, "distance + 1"]       # Uses parameter `distance`
    orientation:
      - "-x0 * myradius / (1 - myradius)"  # Complex expression (quotes recommended)
      - "-x1 * myradius / (1 - myradius)"
      - 0
    torque: [k*x0, k*x1, 0]
"""

In [4]:
myplankton= SoftPlankton(yaml_data, verbose=True)

  Found variables: x0, x1, distance, myradius
    Sphere 0
      Radius: myradius
      Position: ['0', '0', 'distance']
      Orientation: ['x0', 'x1', '0']
      Force: ['0', '0', '0']
      Torque: ['-k*x0', '-k*x1', '0']
    Sphere 1
      Radius: 1 - myradius
      Position: ['0', '0', 'distance + 1']
      Orientation: ['-myradius*x0/(1 - myradius)', '-myradius*x1/(1 - myradius)', '0']
      Force: ['0', '0', '0']
      Torque: ['k*x0', 'k*x1', '0']


## Computing useful tensors

In the full problem, we use the tensors $\mathbb{V}$, $\mathbb{M}$ and $\mathbb{G}$ such that

$$\dot{\mathbf{Q}} = \frac{1}{\mu}\mathbb{M}\cdot\mathbf{F} + \mathbb{V}\cdot \mathbf{u}_\infty + \mathbb{G}\cdot\mathbf{S}_\infty$$

with

$$\dot{\mathbf{Q}} =
    \begin{bmatrix}
        \dot{\mathbf{q}} \\
        \mathbf{u}_0
    \end{bmatrix}
$$

By default the grand force vector $\mathbf{F}$ is written with the origin of the framework of the particle as the reference point for torques.

Alternatively, one can use $\tilde{\mathbb{M}}$ if the forces and torques $\tilde{\mathbf{F}}$ are applied at the center of each sphere.

In the particular case of a parallel force, like buoyancy, we can simplify a lot providing a mass vector gathering the relative mass of all spheres (with respect to the density of the surrounding fluid). In that case:

$$\mathbb{M}\cdot\mathbf{F} = \mathbb{M}_m\cdot\mathbf{g},$$

with $\mathbf{g}$ the acceleration of gravity.

In [11]:

M, Mtilde, Mm, G, V = myplankton.compute_full_mobility_problem()
print("\nVelocity Matrix:")
print(V)
print("\nMobility Matrix M (multiplied by mu):")
print(M)
print("\nMobility Matrix Mm (multiplied by mu):")
print(Mm)
print("\nStrain Mobility Matrix:")
print(G)


Velocity Matrix:
[[ 0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.]
 [ 1.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.]
 [ 0.  0.  0.  0.  1.  0.]
 [ 0.  0.  0.  0.  0.  1.]]

Mobility Matrix M (multiplied by mu):
[[ 0.     0.15   0.     1.147  0.     0.     0.    -0.167  0.    -0.12   0.     0.   ]
 [-0.15   0.     0.     0.     1.147  0.     0.167  0.     0.     0.    -0.12   0.   ]
 [ 0.154  0.     0.     0.    -0.25   0.     0.105  0.     0.     0.    -0.051  0.   ]
 [ 0.     0.154  0.     0.25   0.     0.     0.     0.105  0.     0.051  0.     0.   ]
 [ 0.     0.     0.07   0.     0.     0.     0.     0.     0.07   0.     0.     0.   ]
 [ 0.     0.101  0.     0.314  0.     0.     0.     0.035  0.     0.052  0.     0.   ]
 [-0.101  0.     0.     0.     0.314  0.    -0.035  0.     0.     0.     0.052  0.   ]
 [ 0.     0.     0.     0.     0.     0.093  0.     0.     0.     0.     0.     0.093]]

Mobility Matrix Mm (multipli

## Fast dynamics

In the limit of fast dynamics of d.o.f., such that we have $\dot{\mathbf{q}}= 0$, the mobility equation reduces formally to a rigid-body mobility equation

$$\mathbf{u}_0 = \frac{1}{\mu}\mathbb{M}_{k}\cdot\mathbf{F} + \mathbf{u}_\infty + \mathbb{G}_{k}\cdot\mathbf{S}_\infty ,$$

but $\mathbb{M}_{k}(\mathbf{q})$ and $\mathbb{G}_{k}(\mathbf{q})$ are non-linear functions in general (but they can be tabulated provided the dimension of $\mathbf{q}$ is small enough), and

$$ \mathbf{q} =  \mathbb{M}_{dof}\cdot\mathbf{F} + \mathbb{G}_{dof}\cdot\mathbf{S}_\infty.$$

If the only external force is buyoancy applied at the sphere centers, such that $\tilde{\mathbf{F}} = \mathbf{T}_m\cdot\mathbf{g}$, the problem reduces to

$$\mathbf{u}_0 = \frac{1}{\mu}\mathbb{M}_{k,m}\cdot\mathbf{g} + \mathbf{u}_\infty + \mathbb{G}_{k}\cdot\mathbf{S}_\infty ,$$

$$ \mathbf{q} =  \mathbb{M}_{dof,m}\cdot\mathbf{g} + \mathbb{G}_{dof}\cdot\mathbf{S}_\infty.$$

In [12]:
#myplankton.set_dof_defaults(new_dict={"x0": 0, "x1": 0})
Mk, Mmean, Mkm, Gk, Mdof, Mdofm, Gdof, x_cr, x_cm, *_ = myplankton.compute_fast_mobility_problem()

In [13]:
print("EQUATION FOR RIGID BODY")
print("\nMobility Matrix Mk (multiplied by mu):")
print(Mk)
print("\nUnique Mobility Matrix Mk (multiplied by mu):")
print(Mmean)
print("\nStrain Mobility Matrix Gk (Sxx, Sxy, Sxz, Syy, Syz):")
print(Gk)

print("\n\nEQUATION FOR DEGREES OF FREEDOM", myplankton.dof_variables)
print("\nMobility Matrix Mdof (multiplied by k):")
print(Mdof)
print("\nStrain Mobility Matrix Gdof (multiplied by k/mu) (Sxx, Sxy, Sxz, Syy, Syz):")
print(Gdof)

EQUATION FOR RIGID BODY

Mobility Matrix Mk (multiplied by mu):
[[ 0.131  0.     0.     0.    -0.07   0.     0.131  0.     0.     0.    -0.07   0.   ]
 [ 0.     0.131  0.     0.07   0.     0.     0.     0.131  0.     0.07   0.     0.   ]
 [ 0.     0.     0.07   0.     0.     0.     0.     0.     0.07   0.     0.     0.   ]
 [ 0.     0.07   0.     0.077  0.     0.     0.     0.07   0.     0.077  0.     0.   ]
 [-0.07   0.     0.     0.     0.077  0.    -0.07   0.     0.     0.     0.077  0.   ]
 [ 0.     0.     0.     0.     0.     0.093  0.     0.     0.     0.     0.     0.093]]

Unique Mobility Matrix Mk (multiplied by mu):
[[ 0.131  0.     0.     0.    -0.07   0.   ]
 [ 0.     0.131  0.     0.07   0.     0.   ]
 [ 0.     0.     0.07   0.     0.     0.   ]
 [ 0.     0.07   0.     0.077  0.     0.   ]
 [-0.07   0.     0.     0.     0.077  0.   ]
 [ 0.     0.     0.     0.     0.     0.093]]

Strain Mobility Matrix Gk (Sxx, Sxy, Sxz, Syy, Syz):
[[ 0.     0.     0.664  0.     0.   ]
 [ 

In [14]:
print("EQUATION FOR RIGID BODY with gravity")
print("\nUnique Mobility Matrix Mk,m (multiplied by mu):")
print(Mkm)
print("\nStrain Mobility Matrix Gk (Sxx, Sxy, Sxz, Syy, Syz):")
print(Gk)

print("\n\nEQUATION FOR DEGREES OF FREEDOM", myplankton.dof_variables)
print("\nMobility Matrix Mdof,m (multiplied by k):")
print(Mdofm)
print("\nStrain Mobility Matrix Gdof (multiplied by k/mu) (Sxx, Sxy, Sxz, Syy, Syz):")
print(Gdof)

EQUATION FOR RIGID BODY with gravity

Unique Mobility Matrix Mk,m (multiplied by mu):
[[ 0.07   0.     0.   ]
 [ 0.     0.07   0.   ]
 [ 0.     0.     0.   ]
 [ 0.     0.077  0.   ]
 [-0.077  0.     0.   ]
 [ 0.     0.     0.   ]]

Strain Mobility Matrix Gk (Sxx, Sxy, Sxz, Syy, Syz):
[[ 0.     0.     0.664  0.     0.   ]
 [ 0.     0.     0.     0.     0.664]
 [-0.967  0.     0.    -0.967  0.   ]
 [ 0.     0.     0.     0.    -0.239]
 [ 0.     0.     0.239  0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]]


EQUATION FOR DEGREES OF FREEDOM ['x0', 'x1']

Mobility Matrix Mdof,m (multiplied by k):
[[ 0.     0.155  0.   ]
 [-0.155  0.     0.   ]]

Strain Mobility Matrix Gdof (multiplied by k/mu) (Sxx, Sxy, Sxz, Syy, Syz):
[[ 0.     0.     0.     0.    -1.034]
 [ 0.     0.     1.034  0.     0.   ]]


In [15]:
myplankton.set_param_defaults(new_dict={
    "XYZcr0": x_cr[0],
    "XYZcr1": x_cr[1],
    "XYZcr2": x_cr[2],
    })

matrices = myplankton.compute_fast_mobility_problem()

print("EQUATION FOR RIGID BODY with gravity")
print("\nUnique Mobility Matrix Mk,m (multiplied by mu):")
print(matrices.Mkm)
print("\nStrain Mobility Matrix Gk (Sxx, Sxy, Sxz, Syy, Syz):")
print(matrices.Gk)

print("\n\nEQUATION FOR DEGREES OF FREEDOM", myplankton.dof_variables)
print("\nMobility Matrix Mdof,m (multiplied by k):")
print(matrices.Mdofm)
print("\nStrain Mobility Matrix Gdof (multiplied by k/mu) (Sxx, Sxy, Sxz, Syy, Syz):")
print(matrices.Gdof)

OLD default param values: ['XYZcr0', 'XYZcr1', 'XYZcr2', 'distance', 'k', 'myradius'] [ 0.    0.    0.    0.    1.    0.25]
NEW default param values: ['XYZcr0', 'XYZcr1', 'XYZcr2', 'distance', 'k', 'myradius'] [ 0.     0.    -0.909  0.     1.     0.25 ]
EQUATION FOR RIGID BODY with gravity

Unique Mobility Matrix Mk,m (multiplied by mu):
[[ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 0.     0.     0.   ]
 [ 0.     0.077  0.   ]
 [-0.077  0.     0.   ]
 [ 0.     0.     0.   ]]

Strain Mobility Matrix Gk (Sxx, Sxy, Sxz, Syy, Syz):
[[ 0.     0.    -0.028  0.     0.   ]
 [ 0.     0.     0.     0.    -0.028]
 [-0.058  0.     0.    -0.058  0.   ]
 [ 0.     0.     0.     0.    -0.239]
 [ 0.     0.     0.239  0.     0.   ]
 [ 0.     0.     0.     0.     0.   ]]


EQUATION FOR DEGREES OF FREEDOM ['x0', 'x1']

Mobility Matrix Mdof,m (multiplied by k):
[[ 0.     0.155  0.   ]
 [-0.155  0.     0.   ]]

Strain Mobility Matrix Gdof (multiplied by k/mu) (Sxx, Sxy, Sxz, Syy, Syz):
[[ 0.     0.    